In [8]:
import duckdb
con = duckdb.connect()
con.execute("ATTACH 'ducklake:sqlite:data/metadata.sqlite' AS lake (DATA_PATH 'data/files/')")
con.execute("USE lake")
df_clean = con.execute(
    "SELECT * " \
    "FROM weather").fetchdf()



In [ ]:
con.execute("SELECT * FROM lake.snapshots();").fetch_df()

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message,commit_extra_info
0,0,2026-04-03 18:28:55.480580+02:00,0,{'schemas_created': ['main']},None,None,None
1,1,2026-04-03 18:28:55.541765+02:00,1,"{'tables_created': ['main.weather'], 'tables_i...",None,None,None


In [12]:
con.execute(
    "CREATE TABLE people (id INTEGER, name VARCHAR, salary FLOAT);"
    "INSERT INTO people VALUES (1, 'John', 92_000.0), (2, 'Anna', 100_000.0);"
    )

In [13]:
con.execute("""
                    MERGE INTO people
                USING (
                    SELECT
                        unnest([3, 1]) AS id,
                        unnest(['Sarah', 'John']) AS name,
                        unnest([95_000.0, 105_000.0]) AS salary
                ) AS upserts
                ON (upserts.id = people.id)
                WHEN MATCHED THEN UPDATE
                WHEN NOT MATCHED THEN INSERT;

            FROM people;""")

In [15]:
con.execute("SELECT * FROM lake.snapshots();").fetchdf()

,snapshot_id,snapshot_time,schema_version,changes,author,commit_message,commit_extra_info
0,0,2026-04-03 18:28:55.480580+02:00,0,{'schemas_created': ['main']},None,None,None
1,1,2026-04-03 18:28:55.541765+02:00,1,"{'tables_created': ['main.weather'], 'tables_i...",None,None,None
2,2,2026-04-03 18:44:08.135538+02:00,2,{'tables_created': ['main.people']},None,None,None
3,3,2026-04-03 18:44:08.150109+02:00,2,{'inlined_insert': ['2']},None,None,None
4,4,2026-04-03 18:44:45.899266+02:00,2,"{'tables_inserted_into': ['2'], 'inlined_delet...",None,None,None


In [22]:
con.execute("SELECT * FROM people AT (VERSION => 4);").fetchdf()

,id,name,salary
0,1,John,105000.0
1,3,Sarah,95000.0
2,2,Anna,100000.0
